In [ ]:
import glob
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

download_folder = os.path.abspath('selenium_tests/downloads')
os.makedirs(download_folder, exist_ok=True)

chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option('prefs', {
    'download.default_directory': download_folder,
    'download.prompt_for_download': False,
    'download.directory_upgrade': True,
    'safebrowsing.enabled': True
})

driver = webdriver.Chrome(
    service=ChromeService(ChromeDriverManager().install()),
    options=chrome_options
)
driver.implicitly_wait(10)
driver.get('http://localhost:3000/#/admin-clearance')

# 1) Admin Clearance Login (normal Sign In does not allow ADMIN accounts)
time.sleep(1)
admin_inputs = driver.find_elements(By.XPATH, "//form//input")
admin_inputs[0].send_keys('HQ-BANGLADESH-SECURITY-2026')
admin_inputs[1].send_keys('admin@sentinelx.gov.bd')
admin_inputs[2].send_keys('demo1234')
driver.find_element(By.XPATH, "//form//button[@type='submit']").click()
time.sleep(3)

# 2) Record existing CSV files before exporting
existing_files = set(glob.glob(os.path.join(download_folder, 'sentinelx_crime_statistics_*.csv')))

# 3) Export the nationwide crime statistics CSV
export_btn = driver.find_element(By.XPATH, "//button[contains(.,'Export Crime Statistics (CSV)')]")
export_btn.click()
time.sleep(3)

# 4) Assert: A new crime-statistics CSV file was downloaded
downloaded_files = set(glob.glob(os.path.join(download_folder, 'sentinelx_crime_statistics_*.csv')))
new_files = downloaded_files - existing_files
assert len(new_files) > 0, 'crime statistics CSV was not downloaded'
print('[OK] Admin Export Crime Stats test passed')
driver.quit()
